### FOMC event study: does KLCI, MGS_10Y, or USDMYR react before (T_pre), on (T_post0), or after (T_post1) the FOMC decision?

Method: for each window-day, a one-sample t-test (and Wilcoxon
signed-rank as a non-parametric check) against H0: mean = 0. Also
tracks the cumulative path (CAR) through the window, to separate
"does it move" from "how much has it moved by when."

In [1]:
import sys
import pandas as pd
from scipy.stats import ttest_1samp, wilcoxon

sys.path.append('..')
from modules.source import Sources, parquet_fomc

ASSETS = [Sources.MGS_10Y, Sources.KLCI, Sources.USDMYR]
WINDOW_ORDER = ["T_pre", "T_post0", "T_post1"]

df = pd.read_parquet(parquet_fomc)
n_events = df["fomc_date"].nunique()
print(f"{n_events} FOMC events, window = {WINDOW_ORDER}\n")


def event_test(x: pd.Series) -> dict:
   x = x.dropna()
   t_stat, t_p = ttest_1samp(x, 0)
   _, w_p = wilcoxon(x)
   return {"mean": x.mean(), "std": x.std(), "n": len(x), "t_stat": t_stat, "t_p": t_p, "wilcoxon_p": w_p}


def add_stars(table: pd.DataFrame) -> pd.DataFrame:
   table["sig"] = table["t_p"].apply(lambda p: "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else "")
   return table


# ---------------------------------------------------------- per-window reaction
print("=" * 70)
print("PER-WINDOW REACTION (mean change, t-test, Wilcoxon)")
print("=" * 70)
rows = []
for asset in ASSETS:
   for w in WINDOW_ORDER:
      res = event_test(df.loc[df["window"] == w, asset])
      res["asset"], res["window"] = asset, w
      rows.append(res)

per_window = add_stars(pd.DataFrame(rows).set_index(["asset", "window"]))
print(per_window[["mean", "std", "n", "t_stat", "t_p", "wilcoxon_p", "sig"]].round(4), "\n")

# --------------------------------------------------------- cumulative path (CAR)
print("=" * 70)
print("CUMULATIVE PATH THROUGH THE WINDOW (running CAR, per event)")
print("=" * 70)
wide = {a: df.pivot(index="fomc_date", columns="window", values=a)[WINDOW_ORDER] for a in ASSETS}

car_rows = []
for asset in ASSETS:
   cum = wide[asset].cumsum(axis=1)  # running sum across T_pre -> T_post0 -> T_post1, per event
   for w in WINDOW_ORDER:
      res = event_test(cum[w])
      res["asset"], res["stage"] = asset, f"through {w}"
      car_rows.append(res)

car = add_stars(pd.DataFrame(car_rows).set_index(["asset", "stage"]))
print(car[["mean", "std", "n", "t_stat", "t_p", "wilcoxon_p", "sig"]].round(4))

72 FOMC events, window = ['T_pre', 'T_post0', 'T_post1']

PER-WINDOW REACTION (mean change, t-test, Wilcoxon)
                   mean     std   n  t_stat     t_p  wilcoxon_p sig
asset   window                                                     
MGS_10Y T_pre   -0.0015  0.0322  72 -0.4028  0.6883      0.5751    
        T_post0 -0.0090  0.0396  72 -1.9354  0.0569      0.0518   *
        T_post1  0.0015  0.0503  72  0.2578  0.7973      0.5476    
KLCI    T_pre    0.0002  0.0093  72  0.1751  0.8615      0.4420    
        T_post0 -0.0010  0.0088  72 -0.9742  0.3332      0.5334    
        T_post1 -0.0002  0.0070  72 -0.3018  0.7637      0.9910    
USDMYR  T_pre   -0.0004  0.0041  72 -0.7561  0.4521      0.9091    
        T_post0 -0.0005  0.0049  72 -0.8320  0.4082      0.9738    
        T_post1 -0.0005  0.0047  72 -0.9418  0.3495      0.5136     

CUMULATIVE PATH THROUGH THE WINDOW (running CAR, per event)
                           mean     std   n  t_stat     t_p  wilcoxon_p sig
asse